# Car Traffic Volume, Toronto (2010-2019)

[Toronto Open Data](https://open.toronto.ca/dataset/traffic-volumes-at-intersections-for-all-modes/) has recorded the total traffic volume in the city of Toronto every decade since the 90's. We are going to process the data from 2010-2019 and look at traffic distributions within the city. The raw traffic data includes all modes of transport and is also segregated based on the movement direction of the vehicles. For simplicity, the data has been reduced such that we only have the total number of cars recorded at a given location irrespective of its movement direction. The post processing of the dataset using pandas is not included in this notebook. The processed .csv file located in CarData folder is imported directly, which contains the necessary location information and the total car volume. We will also be using the [Toronto Community Council Boundaries](https://open.toronto.ca/dataset/community-council-boundaries/) shapefile to visulaise the data using the Folium Choropleth map. This geospatial data is in the EPSG -4326 format. 



In [1]:
import os
import time
import pandas as pd 
import geopandas as gpd
import folium
from folium.plugins import HeatMap
from folium.plugins import MarkerCluster

## read the dataset file as a dataframe 
CTV_Main_Dataset = pd.read_csv('Data/CTV2019_Processed.csv')

## Read the Toronto boundary shapefile through geopandas
Toronto = gpd.read_file('Data/Community Council Boundaries Data - 4326/Community Council Boundaries Data.shp')

## Concatenate the Traffic Volume and Toronto Boundary dataframes  
index = [0,2,1,3]
Choropleth_Dataset = CTV_Main_Dataset.groupby('AREA_CODE',as_index = False)[['TOTAL_CARS']].sum()
Choropleth_Dataset['ind'] = index
Choropleth_Dataset.set_index('ind', inplace=True)
Choropleth_Dataset = pd.concat([Toronto,Choropleth_Dataset],axis=1).reindex(Toronto.index)


## Choropleth Map

In [2]:
## Setting up the title for the map
title_text = 'Car Traffic Volume for Toronto Community Councils (2010-2019)'

title_html = '''
             <h3 align="center" style="font-size:18px"><b>{}</b></h3>
             '''.format(title_text) 

## base map
choromap = folium.Map(location=[43.71, -79.37],
               zoom_start=11)
choromap.get_root().html.add_child(folium.Element(title_html))

bins = list(Choropleth_Dataset['TOTAL_CARS'].quantile([0,0.25,0.5,0.75,1]))

## Using the Folium plugin for the Choropleeth map overlay
folium.Choropleth(geo_data=Choropleth_Dataset,
                  name = 'Choropleth',
                  data=Choropleth_Dataset,
                  columns=['FIELD_5','TOTAL_CARS'],
                  fill_color='PuBu',
                  key_on='feature.properties.AREA_CODE',
                  line_opacity=1,
                  bins = bins,
                  fill_opacity=0.7,
                  overlay=True).add_to(choromap)
folium.LayerControl().add_to(choromap)

## Tooltip styling
style_function = lambda x: {'fillColor': '#ffffff', 
                            'color':'#000000', 
                            'fillOpacity': 0.1, 
                            'weight': 0.1}
highlight_function = lambda x: {'fillColor': '#000000', 
                                'color':'#000000', 
                                'fillOpacity': 0.50, 
                                'weight': 0.1}

tooltip = folium.features.GeoJson(Choropleth_Dataset,
                              style_function=style_function,
                              control=False,
                              highlight_function=highlight_function,
                              tooltip=folium.features.GeoJsonTooltip(fields=['FIELD_7','FIELD_5','TOTAL_CARS'],
                                           aliases=['Council Name','Area Code','Car Traffic Volume (2010-2019)'],
                                           style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"))

                              )
choromap.add_child(tooltip)
choromap.keep_in_front(tooltip)

#show Choropleth Map
choromap



The council boundaries divide Toronto into 4 major areas : 
1. Toronto and East York Community Council (SO)
2. Scarborough Community Council (EA)
3. North York Community Council (NO)
4. Etobicoke York Community Council (WE)

The map shows the total car traffic volume in 10 years (2010-2019) for each of the councils. The volume is in the order of 10 million and is the highest in the Toronto and the East York region, followed by North York. The lower traffic volumes are seen in Etobicoke and Scarborough regions. 

## Heatmap

To pin-point the major traffic hotspots across the city, we need a more continuous representation of the traffic volume data. The Choropleth map provided us the averaged out quantitative data over a large area (for the council boundaries), but if we want to visulaise the data at a smaller scale, we can make use of the heatmap. In the current dataset, there are about 4000 datapoints which help in effective rendering of the heatmap. 

In [3]:
## Setting up the title for the map
title_text = 'Toronto Car Traffic Volume Heatmap (2010-2019)'

title_html = '''
             <h3 align="center" style="font-size:18px"><b>{}</b></h3>
             '''.format(title_text) 

## creating a color map for the heatmap gradient
custom_color_map = {0.2: '#00E100',
                    0.6: '#FFF400',
                    0.9: '#FF4200'}  

heatmap1 = folium.Map(location=[43.7, -79.37],
               zoom_start=11)

heatmap1.get_root().html.add_child(folium.Element(title_html))

HeatMap(CTV_Main_Dataset[['LAT','LON','TOTAL_CARS']], 
        min_opacity=0.3,
        radius = 5,
        gradient = custom_color_map,
        blur = 1).add_to(folium.FeatureGroup(name='Heatmap').add_to(heatmap1))
heatmap1

The heatmap is colored based on the influence of local points and is also sensitive to the zoom level of the map. On the map above, the greener area shows that there is a low traffic volume at that location and the scale increases as we move towards yellow and red. This map clearly highlights the heavy traffic volume near Downtown and its surrounding areas.   

## Cluster Markers
   

In [4]:
title_text = 'Toronto Car Traffic Data Points (2010-2019)'

title_html = '''
             <h3 align="center" style="font-size:18px"><b>{}</b></h3>
             '''.format(title_text) 

MarkerMap = folium.Map(location=[43.7, -79.37],
               zoom_start=10)
MarkerMap.get_root().html.add_child(folium.Element(title_html))

MarkerCluster(locations=CTV_Main_Dataset[['LAT','LON']],
              popups=CTV_Main_Dataset['TOTAL_CARS'].tolist(),
              overlay=True,
              control=True).add_to(folium.FeatureGroup(name='Cluster')).add_to(MarkerMap)
MarkerMap

Finally, we use the cluster markers as it is an efficient way of showing data clusters when the number of datapoints are high. In our dataset, we will use the ClusterMarkers to provide the exact location of the car traffic volume datapoint and add the total cars recorded at that point in the pop-up. This map can be used to find the exact traffic volume for a particular location. This map also tells us the number of datapoints in the area of interest.